# D3DR: Diffusion Models are Secretly Zero-Shot 3DGS Harmonizers

Hello!

In this tutorial we insert an object 3DGS into a scene 3DGS and fix appearance using [D3DR](https://www.norange.io/projects/diff_relight/).

## Table of Contents
1. [Installation](#installation)
2. [Train Object & Scene 3DGS](#get-object-scene-3dgs)
3. [Train D3DR](#train-d3dr)
4. [Plot Results](#plot-results)


## 1. Installation <a id="installation"></a>

In [ ]:
!wget -qO- https://astral.sh/uv/install.sh | sh

In [ ]:
!git clone https://github.com/sevashasla/D3DR

In [ ]:
!cd D3DR/ && uv sync

In [ ]:
# install IC-Light checkpoints
!mkdir D3DR/checkpoints
!wget https://huggingface.co/lllyasviel/ic-light/resolve/main/iclight_sd15_fbc.safetensors -P D3DR/checkpoints
!wget https://huggingface.co/lllyasviel/ic-light/resolve/main/iclight_sd15_fc.safetensors -P D3DR/checkpoints

## 2. Get Object & Scene 3DGS <a id="get-object-scene-3dgs"></a>

In [ ]:
# install data (bathroom_1)
# https://drive.google.com/file/d/1dtPtgSbMdPDTijLuZV50rCQ8xUxzjowT/view?usp=sharing
!gdown 1dtPtgSbMdPDTijLuZV50rCQ8xUxzjowT
!mkdir /content/processed/
!unzip -q /content/bathroom_1.zip -d /content/processed/

We train object 3DGS and scene 3DGS from scratch. However, one can use the checkpoints for [object](https://drive.google.com/file/d/1BcargyfEST6abaA5vZZsFpcrUJYCUjc-/view?usp=sharing) and [scene](https://drive.google.com/file/d/11kVlY6WYd_F60FrOQ8zAjGtN0NXAGhH7/view?usp=sharing), but need to specify nerfstudio configs.

In [ ]:
%cd /content/D3DR
!source .venv/bin/activate; which ns-train

In [ ]:
%env CUDA_HOME=/usr/local/cuda
%env TORCHDYNAMO_DISABLE=1

In [ ]:
!export LC_ALL="en_US.UTF-8"
!export LD_LIBRARY_PATH="/usr/lib64-nvidia"
!export LIBRARY_PATH="/usr/local/cuda/lib64/stubs"
!ldconfig /usr/lib64-nvidia

In [ ]:
# create an object 3DGS
!mkdir /content/raw_3dgs
!uv run ns-train splatfacto \
    --data /content/processed/bathroom_1/obj/ \
    --output-dir /content/raw_3dgs/bathroom_1-obj \
    --pipeline.model.background-color black \
    --viewer.quit-on-train-completion True nerfstudio-data \
    --orientation-method none --center-method none --auto-scale-poses False

In [ ]:
# create a scene 3DGS
!uv run ns-train splatfacto \
    --data /content/processed/bathroom_1/scene_eval/ \
    --output-dir /content/raw_3dgs/bathroom_1-scene_eval \
    --pipeline.model.background-color black \
    --viewer.quit-on-train-completion True nerfstudio-data \
    --orientation-method none --center-method none --auto-scale-poses False

## 3. Train D3DR <a id="train-d3dr"></a>

In [ ]:
# create a config
import json
with open("/content/D3DR/scenes_info.json", "w") as f:
    json.dump(
        {
            "bathroom_1": {
                "obj_desc": "caution wet floor sign",
                "ic_light_prompt": "a yellow plastic sign",
                "scene_desc": "bathroom"
            }
        },
        f,
        indent=4,
    )


In [ ]:
!uv run python3 train_everything.py \
    --exp_name "exp" \
    --root /content/outputs/ \
    --dataset_root /content/processed \
    --scene_name bathroom_1 \
    --gaussian_splatting_root /content/raw_3dgs/ \
    --use_wandb 0

# 4. Plot the results <a id="plot-results"></a>

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

image_dir = "/content/outputs/exp_bathroom_1_000/renderings/obj_scene"

image_files = [
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".gif", ".webp"))
]

random_images = random.sample(image_files, 16)

plt.figure(figsize=(10, 10))

for i, img_path in enumerate(random_images):
    img = Image.open(img_path)

    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(os.path.basename(img_path), fontsize=8)

plt.tight_layout()
plt.show()